In [5]:
import pandas as pd
import re

def clean_headlines(input_file, output_file):
    df = pd.read_csv(
        input_file,
        engine='python',
        on_bad_lines='skip'   # prints which lines are bad instead of crashing
    )
    original_count = len(df)
    
    # 1. Drop duplicates on URL
    df = df.drop_duplicates(subset='url')
    
    # 2. Drop relative/unparseable dates
    relative_date_pattern = r'(yesterday|today|\d+ days? ago|hours? ago|ago)'
    df = df[~df['date'].str.lower().str.contains(relative_date_pattern, na=True)]
    
    # 3. Parse dates and filter strictly to 2022–2024
    df['date'] = pd.to_datetime(df['date'], errors='coerce')
    df = df.dropna(subset=['date'])
    df = df[df['date'].dt.year.between(2022, 2024)]
    
    # 4. Drop error rows
    error_patterns = ['error establishing', 'database connection', '404', 'page not found']
    mask = df['headline'].str.lower().str.contains('|'.join(error_patterns), na=False)
    df = df[~mask]
    
    # 5. Drop headlines that are too short to be meaningful
    df = df[df['headline'].str.len() >= 20]
    
    # 6. Normalize source names
    source_map = {
        'thecitizen': 'The Citizen',
        'dailynews': 'Daily News',
        'ippmedia': 'IPP Media'
    }
    df['source'] = df['source'].str.strip()
    
    # 7. Format date back to string
    df['date'] = df['date'].dt.strftime('%Y-%m-%d')
    
    # 8. Add year_month column for aggregation later
    df['year_month'] = pd.to_datetime(df['date']).dt.to_period('M').astype(str)
    
    print(f"Original: {original_count} rows")
    print(f"After cleaning: {len(df)} rows")
    print(f"Dropped: {original_count - len(df)} rows")
    print(f"\nDate range: {df['date'].min()} → {df['date'].max()}")
    print(f"\nBy source:\n{df['source'].value_counts()}")
    print(f"\nBy year_month (first 6):\n{df['year_month'].value_counts().sort_index().head(6)}")
    
    df.to_csv(output_file, index=False)
    return df


In [6]:
df_clean = clean_headlines('final_news_scrape.csv', 'tz_headlines_clean.csv')

Original: 5698 rows
After cleaning: 5684 rows
Dropped: 14 rows

Date range: 2022-01-01 → 2024-12-31

By source:
source
Daily News     3947
The Citizen    1737
Name: count, dtype: int64

By year_month (first 6):
year_month
2022-01    79
2022-02    58
2022-03    78
2022-04    70
2022-05    75
2022-06    90
Name: count, dtype: int64


C:\Users\K2\AppData\Local\Temp\ipykernel_9612\4145586075.py:5: ParserWarning: Skipping line 1742: Expected 5 fields in line 1742, saw 9

  df = pd.read_csv(
C:\Users\K2\AppData\Local\Temp\ipykernel_9612\4145586075.py:17: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  df = df[~df['date'].str.lower().str.contains(relative_date_pattern, na=True)]


In [8]:
# Check how many rows loaded vs what you expected
print(f"Loaded: {len(df_clean)} rows")

# Also peek at the raw file around line 1742 to see what broke
import subprocess
# In Jupyter, use this to see the problem lines
with open('final_news_scrape.csv', 'r', encoding='utf-8') as f:
    lines = f.readlines()
    print("Line 1741:", lines[1740])  # 0-indexed
    print("Line 1742:", lines[1741])
    print("Line 1743:", lines[1742])

Loaded: 5684 rows
Line 1741: 2021-12-31,What Sh12 billion vanilla investment in Zanzibar means,https://www.thecitizen.co.tz/tanzania/news/national/what-sh12-billion-vanilla-investment-in-zanzibar-means-3668384,The Citizen,2026-05-20

Line 1742: 2021-12-31,Economy on rebound in 2021 amid Omicron variant fears,https://www.thecitizen.co.tz/tanzania/news/national/economy-on-rebound-in-2021-amid-omicron-variant-fears-3668360,The Citizen,2026-05-202024-12-31,Redefining personal tax in Tanzania: A path to innovation and inclusion (Part 2),https://dailynews.co.tz/redefining-personal-tax-in-tanzania-a-path-to-innovation-and-inclusion-part-2/,Daily News,2026-05-20

Line 1743: 2024-12-31,"Goods, services imports rise by 2.3 per cent",https://dailynews.co.tz/goods-services-imports-rise-by-2-3-per-cent/,Daily News,2026-05-20

